# Train a Custom "Coda" Wake Word Model

This notebook trains an openWakeWord model for the wake word **"Coda"**.

It replaces the original Colab notebook's broken piper-phonemize dependency with **edge-tts** (Microsoft Edge TTS) for synthetic clip generation.

**Steps:**
1. Install dependencies
2. Download training data (background noise, negative features)
3. Generate synthetic "coda" clips with edge-tts
4. Augment clips with noise and reverb
5. Train the model
6. Download the trained model

**Time:** ~1 hour for clip generation, ~30 min augmentation, ~2-4 hours training.

**Requirements:** Free Google Colab account. No GPU needed for clip generation, but training is faster with one.

Go to **Runtime > Change runtime type > T4 GPU** before starting.

## Step 1: Install Dependencies

In [ ]:
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

!pip install edge-tts pydub nest_asyncio
!pip install mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0 torch-audiomentations==0.11.0
!pip install acoustics==0.2.6 pronouncing==0.2.0
!pip install datasets==2.14.6 deep-phonemizer==0.0.19

import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# Patch torchaudio if needed (Colab ships torchaudio 2.10+ which removed set_audio_backend)
import torchaudio
if not hasattr(torchaudio, 'set_audio_backend'):
    torchaudio.set_audio_backend = lambda x: None

print("\n=== Setup complete ===")

## Step 2: Download Training Data

This downloads:
- Room impulse responses (for reverb augmentation)
- Background noise and music (AudioSet + FMA)
- Pre-computed negative features (2000 hrs of non-wake-word audio)

Takes ~15-20 minutes.

In [ ]:
import os
import numpy as np
import scipy
import datasets
from pathlib import Path
from tqdm import tqdm

# Room impulse responses
output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for row in tqdm(rir_dataset, desc="Room impulse responses"):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    print(f"Saved RIRs to {output_dir}")
else:
    print(f"RIRs already exist at {output_dir}, skipping")

In [ ]:
# Background noise: AudioSet + Free Music Archive

if not os.path.exists("audioset"):
    os.mkdir("audioset")
fname = "bal_train09.tar"
out = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
if not os.path.exists(out):
    !wget -q -O {out} {link}
    !cd audioset && tar -xf {fname}

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset, desc="AudioSet"):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
    for i in tqdm(range(120), desc="FMA music"):
        row = next(fma_dataset)
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

print("Background audio ready")

In [ ]:
# Pre-computed negative features (~2GB download)
if not os.path.exists("openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
    print("Downloaded training features")

if not os.path.exists("validation_set_features.npy"):
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
    print("Downloaded validation features")

print("All data ready")

## Step 3: Generate Synthetic "Coda" Clips with Edge TTS

This replaces the broken piper-sample-generator step.

Uses Microsoft Edge's free TTS service with 300+ English voices at varied speeds and pitches to create diverse training clips.

Takes ~30-60 minutes for 1500 clips.

In [ ]:
import asyncio
import random
import shutil
from pathlib import Path
from pydub import AudioSegment
import edge_tts
import nest_asyncio
nest_asyncio.apply()

PHRASE = "coda"
N_TRAIN = 1500
N_VAL = 500
TOTAL = N_TRAIN + N_VAL

VARIATIONS = ["coda", "Coda", "CODA", "coda.", "Coda!"]
RATES = ["-30%", "-20%", "-10%", "+0%", "+10%", "+20%", "+30%"]
PITCHES = ["-10Hz", "-5Hz", "+0Hz", "+5Hz", "+10Hz"]

out_train = Path("my_custom_model/positive_clips")
out_val = Path("my_custom_model/positive_clips_val")
tmp = Path("my_custom_model/_tmp")
for d in [out_train, out_val, tmp]:
    d.mkdir(parents=True, exist_ok=True)

voices = [v for v in asyncio.run(edge_tts.list_voices()) if v["Locale"].startswith("en-")]
print(f"{len(voices)} English voices available")

async def generate_clip(text, voice, rate, pitch, path):
    c = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
    await c.save(str(path))

generated = 0
errors = 0

for i in range(TOTAL):
    v = random.choice(voices)
    mp3 = tmp / f"{i:05d}.mp3"
    dest = out_val if i >= N_TRAIN else out_train
    wav = dest / f"{i:05d}.wav"
    try:
        asyncio.run(generate_clip(
            random.choice(VARIATIONS),
            v["Name"],
            random.choice(RATES),
            random.choice(PITCHES),
            mp3
        ))
        audio = AudioSegment.from_mp3(str(mp3))
        audio = audio.set_frame_rate(16000).set_channels(1).set_sample_width(2)
        audio.export(str(wav), format="wav")
        mp3.unlink()
        generated += 1
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f"  skip {v['Name']}: {e}")
        continue

    if (i + 1) % 100 == 0:
        label = "val" if i >= N_TRAIN else "train"
        print(f"  [{label}] {i+1}/{TOTAL} done")

shutil.rmtree(tmp, ignore_errors=True)
n_train = len(list(out_train.glob("*.wav")))
n_val = len(list(out_val.glob("*.wav")))
print(f"\nDone: {n_train} train clips, {n_val} val clips, {errors} errors")

## Step 4: Configure Training

In [ ]:
import yaml

config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

config["target_phrase"] = ["coda"]
config["model_name"] = "coda"
config["n_samples"] = len(list(Path("my_custom_model/positive_clips").glob("*.wav")))
config["n_samples_val"] = len(list(Path("my_custom_model/positive_clips_val").glob("*.wav")))
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as f:
    yaml.dump(config, f)

print(f"Config saved. Training {config['n_samples']} clips for {config['steps']} steps.")
print(f"Phrase: {config['target_phrase']}")

## Step 5: Augment Clips

Mixes the synthetic clips with background noise, reverb, and volume changes.

Takes ~5-15 minutes.

In [ ]:
import sys

# Ensure torchaudio patch is active
import torchaudio
if not hasattr(torchaudio, 'set_audio_backend'):
    torchaudio.set_audio_backend = lambda x: None

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

## Step 6: Train the Model

Trains a small DNN on top of the openWakeWord embedding model.

Takes ~2-4 hours on a T4 GPU, longer on CPU.

In [ ]:
import sys

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

## Step 7: Download Your Model

The trained model is saved as `my_custom_model/coda.onnx`.

Download it and place it in `models/wake-word/coda.onnx` in your voice-claude project, then set `WAKE_WORD_MODEL=/models/coda.onnx` in your `.env` file.

In [ ]:
from pathlib import Path

onnx_path = Path("my_custom_model/coda.onnx")
tflite_path = Path("my_custom_model/coda.tflite")

if onnx_path.exists():
    size = onnx_path.stat().st_size / 1024
    print(f"Model ready: {onnx_path} ({size:.0f} KB)")
    print("Right-click the file in the left sidebar > Download")
else:
    print("ERROR: Model not found. Check the training output above for errors.")

if tflite_path.exists():
    print(f"TFLite version also available: {tflite_path}")
else:
    print("TFLite conversion not available (expected on Python 3.12). Use the .onnx file instead.")

# Quick test
try:
    from openwakeword.model import Model
    model = Model(wakeword_models=[str(onnx_path)])
    print(f"\nModel loads successfully. Detected models: {list(model.models.keys())}")
except Exception as e:
    print(f"Model load test failed: {e}")